In [ ]:
!pip uninstall -y diffusers transformers accelerate huggingface_hub peft

!pip install --no-cache-dir \
  diffusers==0.29.2 \
  transformers==4.41.2 \
  accelerate==0.31.0 \
  huggingface_hub==0.23.4 \
  peft>=0.11.1 \
  safetensors==0.4.3 \
  fastapi uvicorn pyngrok pillow

In [ ]:
%%writefile app.py
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from diffusers import AutoPipelineForText2Image
import torch
import base64
from io import BytesIO

app = FastAPI()

# Allow all origins (so your frontend can call this freely)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

print("Loading SDXL-Turbo model...")

pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")

# Memory optimizations — critical for free Colab T4
pipe.enable_attention_slicing()
pipe.vae.enable_tiling()

print("SDXL-Turbo loaded successfully!")

# Fashion-specific keywords automatically appended to every prompt
FASHION_SUFFIX = (
    "fashion design, haute couture, detailed fabric texture, "
    "professional clothing sketch, runway style, high resolution"
)

NEGATIVE_PROMPT = (
    "blurry, low quality, distorted, deformed, bad anatomy, "
    "watermark, text, ugly, duplicate, morbid, mutilated"
)

class Prompt(BaseModel):
    text: str
    steps: int = 4          # SDXL-Turbo works best at 1–4 steps
    guidance_scale: float = 0.0  # Must be 0.0 for Turbo (it's distilled)

@app.get("/")
def health_check():
    return {"status": "Fashion AI is running 🎨"}

@app.post("/generate")
def generate(prompt: Prompt):
    if not prompt.text.strip():
        raise HTTPException(status_code=400, detail="Prompt cannot be empty.")

    # Clamp steps to safe range for Turbo
    steps = max(1, min(prompt.steps, 4))

    enhanced_prompt = f"{prompt.text.strip()}, {FASHION_SUFFIX}"

    try:
        image = pipe(
            prompt=enhanced_prompt,
            negative_prompt=NEGATIVE_PROMPT,
            num_inference_steps=steps,
            guidance_scale=0.0,   # Turbo ignores CFG — must stay 0.0
        ).images[0]
    except RuntimeError as e:
        raise HTTPException(status_code=500, detail=f"Generation failed: {str(e)}")

    buffered = BytesIO()
    image.save(buffered, format="PNG")
    img_str = base64.b64encode(buffered.getvalue()).decode()

    return {
        "image": img_str,
        "prompt_used": enhanced_prompt,
        "steps": steps
    }

In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("Paste your token here");

In [ ]:
import subprocess

subprocess.Popen([
    "uvicorn", "app:app",
    "--host", "0.0.0.0",
    "--port", "8000"
])

In [ ]:
public_url = ngrok.connect(8000)
print(public_url)